In [2]:
import pandas as pd
import requests
import os
import time
import json
import urllib.parse
from collections import Counter

# --- CONFIGURATION ---
CSV_FILE_PATH = 'species_citation_log.csv'
DOI_COLUMN_NAME = 'paper_doi'
YOUR_EMAIL = "gmarais@ufl.edu"  # <--- PUT YOUR EMAIL HERE
CHECKPOINT_FILE = 'journal_counts_checkpoint.json'
PROCESSED_FILE = 'processed_dois.txt'
CHUNK_SIZE = 500
# --- END CONFIGURATION ---

def load_checkpoint():
    """Loads progress from checkpoint files."""
    journal_counter = Counter()
    processed_dois = set()

    if os.path.exists(CHECKPOINT_FILE):
        try:
            with open(CHECKPOINT_FILE, 'r') as f:
                journal_counter = Counter(json.load(f))
            print(f"Loaded {len(journal_counter)} journal counts from checkpoint.")
        except json.JSONDecodeError:
            print("Could not read checkpoint file, starting fresh.")
            
    if os.path.exists(PROCESSED_FILE):
        with open(PROCESSED_FILE, 'r') as f:
            processed_dois = set(line.strip() for line in f)
        print(f"Loaded {len(processed_dois)} processed DOIs from file.")
        
    return journal_counter, processed_dois

def save_checkpoint(journal_counter, doi_to_process, processed_dois_file):
    """Saves progress to checkpoint files."""
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(journal_counter, f, indent=4)
    processed_dois_file.write(f"{doi_to_process}\n")

def clean_doi(doi_str):
    """Removes URL prefixes from a DOI string."""
    doi_str = str(doi_str).strip()
    if doi_str.startswith('https://doi.org/'):
        return doi_str[16:]
    if doi_str.startswith('http://doi.org/'):
        return doi_str[15:]
    if doi_str.startswith('doi.org/'):
        return doi_str[8:]
    if doi_str.startswith('doi:'):
        return doi_str[4:]
    return doi_str # Assume it's a clean DOI

def main():
    if not YOUR_EMAIL or YOUR_EMAIL == "your_email@ufl.edu":
        print("ERROR: Please edit the script and set the YOUR_EMAIL variable.")
        return

    headers = {
        'User-Agent': f'UF-PhD-Research (mailto:{YOUR_EMAIL})',
        'Accept': 'application/json'
    }
    
    session = requests.Session()
    session.headers.update(headers)

    journal_counter, processed_dois = load_checkpoint()
    
    print(f"--- Starting journal analysis for {CSV_FILE_PATH} ---")

    with open(PROCESSED_FILE, 'a', buffering=1) as processed_file: # Use line buffering
        try:
            for chunk_num, chunk in enumerate(pd.read_csv(CSV_FILE_PATH, chunksize=CHUNK_SIZE)):
                
                unique_dois_original = chunk[DOI_COLUMN_NAME].dropna().unique()
                
                for original_doi_str in unique_dois_original:
                    if original_doi_str in processed_dois:
                        continue 
                        
                    # --- NEW CLEANING STEP ---
                    clean_doi_str = clean_doi(original_doi_str)
                    
                    if not clean_doi_str:
                        processed_dois.add(original_doi_str)
                        save_checkpoint(journal_counter, original_doi_str, processed_file)
                        continue # Skip empty DOIs

                    safe_doi = urllib.parse.quote(clean_doi_str)
                    url = f"https://api.crossref.org/works/{safe_doi}"
                    
                    try:
                        response = session.get(url, timeout=10)

                        if response.status_code == 200:
                            data = response.json()
                            journal_name = data.get('message', {}).get('container-title', [None])[0]
                            
                            if journal_name:
                                journal_counter[journal_name] += 1
                            else:
                                journal_counter["Journal Not Found"] += 1
                        
                        elif response.status_code == 404:
                            # print(f"NOT FOUND: {clean_doi_str}")
                            journal_counter["DOI Not Found in CrossRef"] += 1
                        
                        else:
                            print(f"HTTP ERROR {response.status_code} for {clean_doi_str}")
                            journal_counter["API Error"] += 1

                    except requests.exceptions.RequestException as e:
                        print(f"NETWORK ERROR for {clean_doi_str}: {e}")
                        journal_counter["Network Error"] += 1
                        # If we have a network error, stop and let the user fix it.
                        print("Stopping due to network error. Fix connection and re-run.")
                        return
                    
                    processed_dois.add(original_doi_str)
                    save_checkpoint(journal_counter, original_doi_str, processed_file)
                    
                    time.sleep(0.05) 

                print(f"--- Processed chunk {chunk_num+1} ({len(processed_dois)} total unique DOIs). Checkpoint saved. ---")

        except Exception as e:
            print(f"An unexpected error occurred: {e}")
            print("Stopping script. Run again to resume from the last checkpoint.")
            return

    print("\n\n--- 🔥 Analysis Complete! 🔥 ---")
    
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(journal_counter, f, indent=4)
    print(f"Final results saved to {CHECKPOINT_FILE}")

    print("\n--- Top 50 Journals ---")
    for journal, count in journal_counter.most_common(50):
        print(f"{journal}: {count}")

if __name__ == "__main__":
    main()

Loaded 7 journal counts from checkpoint.
Loaded 156 processed DOIs from file.
--- Starting journal analysis for species_citation_log.csv ---
An unexpected error occurred: list index out of range
Stopping script. Run again to resume from the last checkpoint.
